# SETU — best deployable model via SeqKD (resumable, any language pair)

Trains the **SeqKD** student (the winning method: BLEU 17.1 vs 11.1 for
reference+DPO at 100k) at scale, quantises it to a ≤200 MB offline ONNX artifact,
tests it, and saves it to your Drive for download.

**Setup:** Runtime → Change runtime type → **GPU**, then **Runtime → Run all**
(Drive auth popup at cell 2). Set `PAIR` in cell 2 for a different language
(default `hin_Deva-eng_Latn`; e.g. `tam_Taml-eng_Latn`, `ben_Beng-eng_Latn`, or
the reverse `eng_Latn-tam_Taml`).

**Resumable — just Run All again after a disconnect.** Everything expensive (data,
the distilled corpus, the trained checkpoint, the quantised model) is saved to
`MyDrive/setu_seqkd_deploy/<PAIR>/` and marked **DONE** only on full success —
per-pair, so training several languages never collides. Cell 2 prints a status
list; already-DONE steps skip in seconds. Greedy distill (`--beams 1`) for speed.

In [ ]:
import torch
print('cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - Runtime > Change runtime type > GPU')
from google.colab import drive
drive.mount('/content/drive')
import os
PAIR  = 'hin_Deva-eng_Latn'                          # <-- change for other languages
LIMIT = 250000                                       # SeqKD scales; 250k = strong deployable target
WORK  = f'/content/drive/MyDrive/setu_seqkd_deploy/{PAIR}'   # per-pair, persists across sessions
os.makedirs(WORK, exist_ok=True)
print(f'pair={PAIR}  limit={LIMIT}\npersisting to {WORK}\n')
for name, m in [('data','.done_data'), ('distill','.done_distill'),
                ('SeqKD train','.done_seqkd'), ('quantize','.done_quantize')]:
    print(f"  [{'DONE' if os.path.exists(f'{WORK}/{m}') else '  - '}] {name}")

In [ ]:
# clone + install + GPU configs; set the pair once in model.yaml; data/ -> Drive
%cd /content
!rm -rf /content/SETU_v2
!git clone https://github.com/GeekyRiolu/SETU_v2.git
%cd /content/SETU_v2/SETU
!pip -q install -e ".[data,teacher,quantize]"
!cp configs/model.gpu.yaml configs/model.yaml
!cp configs/training.gpu.yaml configs/training.yaml
!sed -i 's/device: cpu/device: cuda/' configs/teacher.yaml
!sed -i "s|^language_pair:.*|language_pair: {PAIR}|" configs/model.yaml   # every setu-* reads it here
import os
os.makedirs(f'{WORK}/data', exist_ok=True)
!rm -rf data && ln -s {WORK}/data data
print('pair set to', PAIR, '| data ->', os.path.realpath('data'))

In [ ]:
# 1) DATA (SeqKD needs no preference pairs) + 2) teacher-distilled corpus (greedy)
import os
if not os.path.exists(f'{WORK}/.done_data'):
    !setu-data --limit {LIMIT + 2000} && touch {WORK}/.done_data
else:
    print('data already saved - skipping')
if not os.path.exists(f'{WORK}/.done_distill'):
    !setu-distill --limit {LIMIT} --batch-size 32 --beams 1 && touch {WORK}/.done_distill
else:
    print('distilled corpus already saved - skipping')
_d = f'data/distilled/{PAIR}/train.jsonl'
assert os.path.exists(_d) and os.path.getsize(_d) > 0, 'distill produced no corpus'
print('distilled rows:', sum(1 for _ in open(_d)))

In [ ]:
# 3) SeqKD training: SFT on teacher targets (no DPO). Eval on real references.
#    On success: report + full checkpoint saved to Drive so a resume skips it.
import os
if not os.path.exists(f'{WORK}/.done_seqkd'):
    !python scripts/train_full.py --train-corpus distilled --skip-dpo --limit {LIMIT} --dev-size 500 \
        && cp checkpoints/{PAIR}/train_report.json {WORK}/report_seqkd.json \
        && rm -rf {WORK}/ckpt && cp -r checkpoints/{PAIR} {WORK}/ckpt \
        && touch {WORK}/.done_seqkd \
        && echo '=== SeqKD trained + checkpoint saved to Drive ===' || echo '=== TRAINING FAILED - see above ==='
else:
    print('SeqKD already trained - restoring checkpoint from Drive')
    !mkdir -p checkpoints && rm -rf checkpoints/{PAIR} && cp -r {WORK}/ckpt checkpoints/{PAIR}
import json
print(json.load(open(f'{WORK}/report_seqkd.json'))['sft_eval'])

In [ ]:
# 4) QUANTISE the SeqKD student -> INT8/INT4 ONNX, deploy under models/, score.
import os
if not os.path.exists(f'{WORK}/.done_quantize'):
    !setu-quantize --student sft \
        && rm -rf {WORK}/models && cp -r models/{PAIR} {WORK}/models \
        && touch {WORK}/.done_quantize \
        && echo '=== quantised + saved to Drive ===' || echo '=== QUANTIZE FAILED - see above ==='
else:
    print('quantize already done - restoring from Drive')
    !mkdir -p models && rm -rf models/{PAIR} && cp -r {WORK}/models models/{PAIR}
!setu-report --offline-proof

In [ ]:
# 5) TEST the deployed model offline (translate a few sources from the corpus)
import sys, json; sys.path.insert(0, 'src')
from setu.inference.engine import InferenceEngine
from setu.config import resolve_language
src_f, tgt_f = PAIR.split('-')
src_iso, tgt_iso = resolve_language(src_f)['iso'], resolve_language(tgt_f)['iso']
eng = InferenceEngine(models_root='models')
print('using trained model:', not eng.is_stub)
for line in list(open(f'data/processed/{PAIR}/train.jsonl'))[:4]:
    s = json.loads(line)['src_text']
    print(f'{s}  ->  {eng.translate(s, src_iso, tgt_iso).translated_text}')

In [ ]:
# 6) SAVE the deployable model (zip) to Drive for download
!cd models && zip -qr {WORK}/setu_{PAIR}_model.zip {PAIR}
print('deployable model saved to:', f'{WORK}/setu_{PAIR}_model.zip')
print('Download from Drive, then locally: unzip into SETU/models/ and run setu_cli.py')